# Marci's WIP

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

%matplotlib inline
plt.rcParams["figure.figsize"] = (10,6)

In [6]:
import keras
import os

# Make sure the folder exists
os.makedirs("data", exist_ok=True)

# Force Keras to use project folder
f_path_1 = "data/train.csv.zip"
url_1 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/train.csv.zip"

if not os.path.exists(f_path_1):
    keras.utils.get_file(
        fname="train.csv.zip",
        origin=url_1,
        cache_dir=".",
        cache_subdir="data"
    )

# Force Keras to use project folder
f_path_2 = "data/test.csv.zip"
url_2 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/test.csv.zip"

if not os.path.exists(f_path_2):
    keras.utils.get_file(
        fname="test.csv.zip",
        origin=url_2,
        cache_dir=".",
        cache_subdir="data"
    )

## Project 1 - NLP and Text Classification

For this project you will need to classify some angry comments into their respective category of angry. The process that you'll need to follow is (roughly):
<ol>
<li> Use NLP techniques to process the training data. 
<li> Train model(s) to predict which class(es) each comment is in.
    <ul>
    <li> A comment can belong to any number of classes, including none. 
    </ul>
<li> Generate predictions for each of the comments in the test data. 
<li> Write your test data predicitions to a CSV file, which will be scored. 
</ol>

You can use any models and NLP libraries you'd like. 

## Training Data

Use the training data to train your prediction model(s). Each of the classification output columns (toxic to the end) is a human label for the comment_text, assessing if it falls into that category of "rude". A comment may fall into any number of categories, or none at all. Membership in one output category is <b>independent</b> of membership in any of the other classes (think about this when you plan on how to make these predictions - it may also make it easier to split work amongst a team...). 

## Test Data

## Output Details, Submission Info, and Example Submission

For this project, please output your predictions in a CSV file. The structure of the CSV file should match the structure of the example below. 

The output should contain one row for each row of test data, complete with the columns for ID and each classification.

Into Moodle please submit:
<ul>
<li> Your notebook file(s). I'm not going to run them, just look. 
<li> Your sample submission CSV. This will be evaluated for accuracy against the real labels; only a subset of the predictions will be scored. 
</ul>

It is REALLY, REALLY, REALLY important the the structure of your output matches the specifications. The accuracies will be calculated by a script, and it is expecting a specific format. 

### Sample Evaluator

The file prediction_evaluator.ipynb contains an example scoring function, scoreChecker. This function takes a sumbission and an answer key, loops through, and evaluates the accuracy. You can use this to verify the format of your submission. I'm going to use the same function to evaluate the accuracy of your submission, against the answer key (unless I made some mistake in this counting function).

In [2]:
#Construct dummy data for a sample output. 
#You won't do this part, you have real data
#Your data should have the same structure, so the CSV output is the same
dummy_ids = ["dfasdf234", "asdfgw43r52", "asdgtawe4", "wqtr215432"]
dummy_toxic = [0,0,0,0]
dummy_severe = [0,0,0,0]
dummy_obscene = [0,1,1,0]
dummy_threat = [0,1,0,1]
dummy_insult = [0,0,1,0]
dummy_ident = [0,1,1,0]
columns = ["id", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
sample_out = pd.DataFrame( list(zip(dummy_ids, dummy_toxic, dummy_severe, dummy_obscene, dummy_threat, dummy_insult, dummy_ident)),
                    columns=columns)
sample_out.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,dfasdf234,0,0,0,0,0,0
1,asdfgw43r52,0,0,1,1,0,1
2,asdgtawe4,0,0,1,0,1,1
3,wqtr215432,0,0,0,1,0,0


In [3]:
#Write DF to CSV. Please keep the "out.csv" filename. Moodle will auto-preface it with an identifier when I download it. 
#This command should work with your dataframe of predictions. 
sample_out.to_csv('output/out.csv', index=False)  

## Grading

The grading for this is split between accuracy and well written code:
<ul>
<li> 75% - Accuracy. The most accurate will get 100% on this, the others will be scaled down from there. 
<li> 25% - Code quality. Can the code be followed and made sense of - i.e. comments, sections, titles. 
</ul>

##### ***What type of vectorization should we use?***
- CountVectorizer()
- TfidfVectorizer()
- Word2Vec()

##### ***What strategies should we try?***
- Stemming
- Lemming
- Dimensionality reduction
    - SVD
    - Truncated SVD
    - PCA

##### ***How to classify?***
- RandomForestClassifier()

# Get started

In [7]:
# Load data

train_df = pd.read_csv('data/train.csv.zip')
train_df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [10]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB


In [8]:
# Load data

test_df = pd.read_csv('data/test.csv.zip')
test_df.sample(5)

,id,comment_text
26364,26365,:This is wrong. Most of Estonians do not speak...
137016,137017,"""==very bad article== \n I just made some chan..."
140250,140251,""" \n\n == Quick favour == \n\n Could you do a ..."
95214,95215,==You're way to bias and not fair play== \n\n ...
4256,4257,"== Red Wings Season pages == \n\n Hey, I just..."


In [12]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153164 entries, 0 to 153163
Data columns (total 2 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            153164 non-null  int64 
 1   comment_text  153164 non-null  object
dtypes: int64(1), object(1)
memory usage: 2.3+ MB


In [9]:
# Basic cleaning

train_df['comment_text'] = (
    train_df['comment_text']
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip())

test_df['comment_text'] = (
    test_df['comment_text']
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip())


In [14]:
train_df.sample(3)

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
148769,51dc914dfa3fecc5,"""::hi, vic. no probs. i'm not on my laptop atm...",0,0,0,0,0,0
68083,b61c39ea28418d5e,florence cassez i feel i should offer my side ...,0,0,0,0,0,0
14367,25f469a447931043,""" parenthesis """"(, )"""" don't work in links can...",0,0,0,0,0,0


In [ ]:
test_df.sample(3)

,id,comment_text
102629,102630,henry hill doesn't really know what he's talki...
111476,111477,""" == signatures == wow that was quite an expla..."
103688,103689,== for the mod of this website that refuse to ...


In [10]:
import nltk

for package in ['stopwords','punkt','wordnet']:
    nltk.download(package)
    
from nltk.corpus import stopwords 
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english')) 

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\msieb\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\msieb\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\msieb\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [11]:
# Custom stop word tokenizer

class swTokenizer(object):
    def __init__(self, stop_words):
        self.stop_words = stop_words
    def __call__(self, doc):
        tokens = word_tokenize(doc)
        filtered_tok = []
        for tok in tokens:
            if tok not in stop_words:
                filtered_tok.append(tok)
        return filtered_tok

In [12]:
# Custom stem word tokenizer

class stemTokenizer(object):
    def __init__(self, stop_words):
        self.stop_words = stop_words
        from nltk.stem import SnowballStemmer
        self.stemmer = SnowballStemmer(language='english')
    def __call__(self, doc):
        tokens = word_tokenize(doc)
        filtered_tok = []
        for tok in tokens:
            if tok not in stop_words:
                filtered_tok.append(self.stemmer.stem(tok))
        return filtered_tok

In [13]:
# Custom lemmatization tokenizer

class lemmaTokenizer(object):
    def __init__(self, stop_words):
        self.stop_words = stop_words
        from nltk.stem import WordNetLemmatizer
        self.lemmatizer = WordNetLemmatizer()
    def __call__(self, doc):
        tokens = word_tokenize(doc)
        filtered_tok = []
        for tok in tokens:
            if tok not in stop_words:
                filtered_tok.append(self.lemmatizer.lemmatize(tok))
        return filtered_tok

### Models

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer

from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn import metrics
from sklearn.metrics import classification_report

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.porter import PorterStemmer

In [15]:
# Split X and y

target = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']

y = train_df[target]
X = train_df['comment_text']
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [16]:
# Model # 1

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
    ('clf', OneVsRestClassifier(LinearSVC()))
    ])

pipe.fit(X_train, y_train)

# Make predictions
y_pred = pipe.predict(X_test)

In [17]:
# Evaluate the model

print(classification_report(y_test, y_pred, target_names=target))

               precision    recall  f1-score   support

        toxic       0.89      0.69      0.78      3853
 severe_toxic       0.50      0.29      0.36       404
      obscene       0.88      0.70      0.78      2128
       threat       0.70      0.29      0.41       109
       insult       0.79      0.62      0.70      1973
identity_hate       0.70      0.28      0.40       368

    micro avg       0.84      0.64      0.73      8835
    macro avg       0.74      0.48      0.57      8835
 weighted avg       0.84      0.64      0.72      8835
  samples avg       0.06      0.06      0.06      8835



c:\Users\msieb\anaconda3\envs\ml\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\msieb\anaconda3\envs\ml\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\msieb\anaconda3\envs\ml\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


#### Model 1 Performance:

In [21]:
y_pred[:5]

array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]])

In [ ]:
# Retrain model with 100% of data

pipe.fit(train_df['comment_text'], train_df[target])

,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


### Final prediction on test_df

In [34]:
# Make predictions

y_pred = pipe.predict(test_df['comment_text'])

In [35]:
# Convert array to df

pred_df = pd.DataFrame(y_pred, columns=target)

In [36]:
# Attach 'id' column (reset index)

final_df = pd.concat([
    test_df['id'].reset_index(drop=True), 
    pred_df.reset_index(drop=True)], 
    axis=1)

In [37]:
final_df.sample(5)

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
5534,5535,0,0,0,0,0,0
128306,128307,0,0,0,0,0,0
18342,18343,0,0,0,0,0,0
39971,39972,0,0,0,0,0,0
52763,52764,1,0,1,0,1,0


In [ ]:
# Make sure everything lines up

len(final_df) == len(test_df)

True

In [46]:
# Check for nulls

test_df['comment_text'].isna().sum()

np.int64(0)

In [43]:
# Export predictions to CSV

final_df.to_csv('output/out.csv', index=False)  